In [1]:
"""
Advanced Spark Transformations & UDFs
Using your actual data: sales, products, users
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window
import time
import re

# Create Spark session
spark = SparkSession.builder \
    .appName("AdvancedTransformations") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark session created: {spark.version}")


print("=" * 60)
print("ADVANCED TRANSFORMATIONS & UDFs")
print("=" * 60)

26/03/12 16:49:51 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/12 16:49:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 16:49:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark session created: 3.5.0
ADVANCED TRANSFORMATIONS & UDFs


In [3]:
# Read all sales, products, users files
# Load your actual data
data_dir = "/home/odinsbeard/Data_engineering_Journey/week6_spark/data/input"

print("\n📂 Loading data...")
sales = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{data_dir}/sales_*.csv")

products = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{data_dir}/products_*.csv")

users = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{data_dir}/users_*.csv")

print(f"✅ Sales: {sales.count():,} rows")
print(f"✅ Products: {products.count():,} rows")
print(f"✅ Users: {users.count():,} rows")

# Register temp views for SQL
sales.createOrReplaceTempView("sales")
products.createOrReplaceTempView("products")
users.createOrReplaceTempView("users")


📂 Loading data...


✅ Sales: 20,000,000 rows
✅ Products: 400,000 rows


✅ Users: 4,000,000 rows


In [4]:
# ────────────────────────────────────────────────────────────────────────────
# 1. COMPLEX COLUMN TRANSFORMATIONS
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("1. COMPLEX COLUMN TRANSFORMATIONS")
print("=" * 60)

# CASE WHEN - Customer segmentation based on spending
print("\n📊 Customer Segmentation based on spending...")

customer_spending = sales.groupBy("user_id") \
    .agg(
        count("*").alias("total_orders"),
        sum("final_amount").alias("total_spent"),
        avg("final_amount").alias("avg_order_value")
    )

customer_segments = customer_spending \
    .withColumn("segment",
        when(col("total_spent") >= 1000, "VIP")
        .when(col("total_spent") >= 500, "Gold")
        .when(col("total_spent") >= 100, "Silver")
        .otherwise("Bronze")
    ) \
    .withColumn("order_frequency",
        when(col("total_orders") >= 20, "Very Frequent")
        .when(col("total_orders") >= 10, "Frequent")
        .when(col("total_orders") >= 5, "Regular")
        .otherwise("Occasional")
    )

print("Customer segments:")
customer_segments.show(10)



# ARRAY operations................
print("\n📊 Working with arrays...")

# Collect products per user
user_products = sales.groupBy("user_id") \
    .agg(
        collect_list("product_id").alias("products_bought"),
        collect_set("country").alias("countries_purchased_from")
    )

print("User purchase arrays:")
user_products.show(10, truncate=False)


# Array operations.................
user_analysis = user_products \
    .withColumn("num_products", size(col("products_bought"))) \
    .withColumn("num_countries", size(col("countries_purchased_from"))) \
    .withColumn("first_product", element_at(col("products_bought"), 1))

print("User analysis with array ops:")
user_analysis.select("user_id", "num_products", "num_countries", "first_product").show(10)


# EXPLODE - Convert arrays to rows
print("\n📊 Exploding arrays - products per user:")

user_products_exploded = user_analysis \
    .select("user_id", explode("products_bought").alias("product_id"))

user_products_exploded.show(10)



1. COMPLEX COLUMN TRANSFORMATIONS

📊 Customer Segmentation based on spending...
Customer segments:


+-------+------------+------------------+------------------+-------+---------------+
|user_id|total_orders|       total_spent|   avg_order_value|segment|order_frequency|
+-------+------------+------------------+------------------+-------+---------------+
| 667046|          10|           4598.94|459.89399999999995|    VIP|       Frequent|
|  65478|          10|           4350.26|           435.026|    VIP|       Frequent|
| 630408|           9|3902.6100000000006| 433.6233333333334|    VIP|        Regular|
| 171142|          14|10292.619999999999| 735.1871428571428|    VIP|       Frequent|
| 304448|          11|           2226.91|202.44636363636363|    VIP|       Frequent|
| 971109|          13| 7833.460000000001| 602.5738461538463|    VIP|       Frequent|
| 718133|          15| 5244.379999999999| 349.6253333333333|    VIP|       Frequent|
| 376168|          11| 6022.439999999999| 547.4945454545453|    VIP|       Frequent|
| 212727|          13| 6449.649999999999|  496.126923076923|    V

+-------+-----------------------------------------------------------------------------------------+------------------------------------------------+
|user_id|products_bought                                                                          |countries_purchased_from                        |
+-------+-----------------------------------------------------------------------------------------+------------------------------------------------+
|26     |[2242, 2335, 83892, 19929, 83845, 71587, 117369, 6807, 189221, 100000]                   |[France, Brazil, Japan, UK]                     |
|27     |[29110, 99148, 59249, 94754, 11517, 141966, 96031, 95553, 50883, 87880, 32310]           |[Australia, Brazil, Germany, Japan, UK, USA]    |
|28     |[19792, 82284, 7646, 282541, 10858, 327405, 373011]                                      |[France, Japan, UK]                             |
|31     |[78684, 69457, 93954, 96211, 365167, 158353, 132677, 37942, 154618, 48514, 69069]        |[German

+-------+------------+-------------+-------------+
|user_id|num_products|num_countries|first_product|
+-------+------------+-------------+-------------+
|     26|          10|            4|         2242|
|     27|          11|            6|        29110|
|     28|           7|            3|        19792|
|     31|          11|            5|        78684|
|     34|          12|            6|        44118|
|     44|           7|            4|        74921|
|     53|           8|            5|        93237|
|     65|          12|            6|        84949|
|     76|           8|            3|        22126|
|     78|           6|            4|        31977|
+-------+------------+-------------+-------------+
only showing top 10 rows


📊 Exploding arrays - products per user:


[Stage 41:>                                                         (0 + 1) / 1]

+-------+----------+
|user_id|product_id|
+-------+----------+
|     12|     36202|
|     12|     46564|
|     12|     23721|
|     12|     78245|
|     12|      1613|
|     22|     17481|
|     22|     74239|
|     22|     41950|
|     22|     78993|
|     22|     57929|
+-------+----------+
only showing top 10 rows



In [5]:
# ────────────────────────────────────────────────────────────────────────────
# 2. PYTHON UDFs (User-Defined Functions)
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("2. USER-DEFINED FUNCTIONS (UDFs)")
print("=" * 60)

# Simple UDF - Email domain extraction
def extract_email_domain(email):
    """Extract domain from email address"""
    if email and '@' in email:
        return email.split('@')[-1]
    return "unknown.com"

# Register UDF
extract_domain_udf = udf(extract_email_domain, StringType())

print("\n📧 Extracting email domains...")
users_with_domain = users \
    .withColumn("email_domain", extract_domain_udf(col("email")))

users_with_domain.select("email", "email_domain").show(10, truncate=False)

# Domain statistics
print("\n📊 Email domain distribution:")
users_with_domain.groupBy("email_domain") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)

# Complex UDF - Product category from product_id
def categorize_product(product_id):
    """Categorize product based on ID range"""
    if product_id < 20000:
        return "Electronics"
    elif product_id < 40000:
        return "Clothing"
    elif product_id < 60000:
        return "Home"
    elif product_id < 80000:
        return "Sports"
    else:
        return "Other"

categorize_udf = udf(categorize_product, StringType())

print("\n🏷️ Categorizing products...")
products_with_category = products \
    .withColumn("derived_category", categorize_udf(col("product_id")))

products_with_category.select("product_id", "product_name", "category", "derived_category").show(10)



2. USER-DEFINED FUNCTIONS (UDFs)

📧 Extracting email domains...


+------------------+------------+
|email             |email_domain|
+------------------+------------+
|user1@example.com |example.com |
|user2@example.com |example.com |
|user3@example.com |example.com |
|user4@example.com |example.com |
|user5@example.com |example.com |
|user6@example.com |example.com |
|user7@example.com |example.com |
|user8@example.com |example.com |
|user9@example.com |example.com |
|user10@example.com|example.com |
+------------------+------------+
only showing top 10 rows


📊 Email domain distribution:


+------------+-------+
|email_domain|  count|
+------------+-------+
| example.com|4000000|
+------------+-------+


🏷️ Categorizing products...
+----------+------------------+-----------+----------------+
|product_id|      product_name|   category|derived_category|
+----------+------------------+-----------+----------------+
|    200001|       Socks Basic|   Clothing|           Other|
|    200002|        Belt Basic|   Clothing|           Other|
|    200003|        Shorts Max|   Clothing|           Other|
|    200004|    Biography Plus|      Books|           Other|
|    200005|     Blanket Basic|       Home|           Other|
|    200006|     Football Lite|     Sports|           Other|
|    200007|      Bicycle Lite|     Sports|           Other|
|    200008|    Dumbbells Lite|     Sports|           Other|
|    200009|     Biography Pro|      Books|           Other|
|    200010|External Drive Pro|Electronics|           Other|
+----------+------------------+-----------+----------------+
o

In [6]:
# ────────────────────────────────────────────────────────────────────────────
# 3. PANDAS UDFs (Vectorized - Much Faster!)
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("3. PANDAS UDFs (Vectorized for Performance)")
print("=" * 60)

from pyspark.sql.functions import pandas_udf
import pandas as pd

# Pandas UDF - Calculate spending tier based on amount
@pandas_udf(StringType())
def spending_tier(amounts: pd.Series) -> pd.Series:
    """Categorize spending amounts into tiers"""
    conditions = [
        (amounts > 1000, "Premium"),
        (amounts > 500, "High"),
        (amounts > 100, "Medium"),
        (amounts > 0, "Low")
    ]
    
    result = pd.Series(["Unknown"] * len(amounts))
    for condition, label in conditions:
        result[condition] = label
    
    return result

print("\n💰 Adding spending tiers to sales...")
sales_with_tiers = sales \
    .withColumn("tier", spending_tier(col("final_amount")))

sales_with_tiers.select("sale_id", "final_amount", "tier").show(10)

# Group by tier
print("\n📊 Sales by tier:")
sales_with_tiers.groupBy("tier") \
    .agg(
        count("*").alias("num_sales"),
        sum("final_amount").alias("total_amount"),
        avg("final_amount").alias("avg_amount")
    ) \
    .orderBy(col("avg_amount").desc()) \
    .show()



3. PANDAS UDFs (Vectorized for Performance)

💰 Adding spending tiers to sales...


+-------+------------+----+
|sale_id|final_amount|tier|
+-------+------------+----+
|      1|      221.24| Low|
|      2|     1090.37| Low|
|      3|      238.29| Low|
|      4|      404.05| Low|
|      5|      539.78| Low|
|      6|     1262.64| Low|
|      7|       40.73| Low|
|      8|      278.16| Low|
|      9|     1244.37| Low|
|     10|      292.11| Low|
+-------+------------+----+
only showing top 10 rows


📊 Sales by tier:


[Stage 48:===================================================>    (11 + 1) / 12]

+----+---------+-------------------+-----------------+
|tier|num_sales|       total_amount|       avg_amount|
+----+---------+-------------------+-----------------+
| Low| 20000000|9.182953898410326E9|459.1476949205163|
+----+---------+-------------------+-----------------+



In [7]:
# ────────────────────────────────────────────────────────────────────────────
# 4. WINDOW FUNCTIONS - Advanced
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("4. ADVANCED WINDOW FUNCTIONS")
print("=" * 60)

# Ranking within partitions - Top products by revenue per country
print("\n🏆 Top 3 products per country by revenue...")

# Calculate product revenue by country
product_country_revenue = sales.groupBy("product_id", "country") \
    .agg(sum("final_amount").alias("revenue"))

# Add product names
product_country_with_names = product_country_revenue \
    .join(products.select("product_id", "product_name", "category"), "product_id")

# Window for ranking within country
window_country = Window.partitionBy("country").orderBy(col("revenue").desc())

# Add ranks
ranked_products = product_country_with_names \
    .withColumn("rank", rank().over(window_country)) \
    .filter(col("rank") <= 3)

ranked_products.orderBy("country", "rank").show(20, truncate=False)

# Running total of sales per day
print("\n📈 Running total of sales per day...")

daily_sales = sales.groupBy("sale_date") \
    .agg(sum("final_amount").alias("daily_total")) \
    .orderBy("sale_date")

window_running = Window.orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

daily_with_running = daily_sales \
    .withColumn("running_total", sum("daily_total").over(window_running))

daily_with_running.show(15)

# LAG/LEAD - Compare with previous day
print("\n📊 Day-over-day comparison...")

window_day = Window.orderBy("sale_date")

daily_comparison = daily_sales \
    .withColumn("prev_day", lag("daily_total", 1).over(window_day)) \
    .withColumn("next_day", lead("daily_total", 1).over(window_day)) \
    .withColumn("change", col("daily_total") - col("prev_day")) \
    .withColumn("pct_change", 
                when(col("prev_day").isNotNull(),
                     round((col("daily_total") - col("prev_day")) / col("prev_day") * 100, 2))
                .otherwise(0))

daily_comparison.show(15)




4. ADVANCED WINDOW FUNCTIONS

🏆 Top 3 products per country by revenue...


26/03/12 17:27:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+---------+------------------+--------------+-----------+----+
|product_id|country  |revenue           |product_name  |category   |rank|
+----------+---------+------------------+--------------+-----------+----+
|40825     |Australia|18351.94          |Sweater Pro   |Clothing   |1   |
|83627     |Australia|18166.68          |Poetry Basic  |Books      |2   |
|59018     |Australia|17559.88          |Skirt Lite    |Clothing   |3   |
|47573     |Brazil   |19322.2           |Textbook Lite |Books      |1   |
|46233     |Brazil   |18885.45          |Fiction Max   |Books      |2   |
|75950     |Brazil   |18733.239999999998|Jacket Lite   |Clothing   |3   |
|99074     |Canada   |20005.49          |Smartphone Max|Electronics|1   |
|37350     |Canada   |19009.849999999995|Table Plus    |Home       |2   |
|45842     |Canada   |17997.71          |Towel Basic   |Home       |3   |
|34715     |France   |18935.68          |Poetry Max    |Books      |1   |
|99920     |France   |18174.95        

26/03/12 17:27:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------------+--------------------+
| sale_date|      daily_total|       running_total|
+----------+-----------------+--------------------+
|2023-01-01|8344250.389999996|   8344250.389999996|
|2023-01-02|8494198.619999994| 1.683844900999999E7|
|2023-01-03|8329097.010000002|2.5167546019999992E7|
|2023-01-04|       8447347.07| 3.361489308999999E7|
|2023-01-05|8325709.419999999| 4.194060250999999E7|
|2023-01-06|8251309.149999999| 5.019191165999999E7|
|2023-01-07|8486690.569999998| 5.867860222999999E7|
|2023-01-08|8430134.309999997|6.7108736539999984E7|
|2023-01-09|8316905.500000002| 7.542564203999999E7|
|2023-01-10|8458165.069999997| 8.388380710999998E7|
|2023-01-11|8220504.190000001| 9.210431129999998E7|
|2023-01-12|8198554.540000001|1.0030286583999999E8|
|2023-01-13|8401967.700000005|1.0870483353999999E8|
|2023-01-14|8396845.359999996|1.1710167889999999E8|
|2023-01-15|       8407760.13|1.2550943902999999E8|
+----------+-----------------+--------------------+
only showing

26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:27:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:28:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:28:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 17:28:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/12 1

+----------+-----------------+-----------------+-----------------+-------------------+----------+
| sale_date|      daily_total|         prev_day|         next_day|             change|pct_change|
+----------+-----------------+-----------------+-----------------+-------------------+----------+
|2023-01-01|8344250.389999996|             NULL|8494198.619999994|               NULL|       0.0|
|2023-01-02|8494198.619999994|8344250.389999996|8329097.010000002| 149948.22999999765|       1.8|
|2023-01-03|8329097.010000002|8494198.619999994|       8447347.07|-165101.60999999195|     -1.94|
|2023-01-04|       8447347.07|8329097.010000002|8325709.419999999| 118250.05999999866|      1.42|
|2023-01-05|8325709.419999999|       8447347.07|8251309.149999999| -121637.6500000013|     -1.44|
|2023-01-06|8251309.149999999|8325709.419999999|8486690.569999998| -74400.26999999955|     -0.89|
|2023-01-07|8486690.569999998|8251309.149999999|8430134.309999997|   235381.419999999|      2.85|
|2023-01-08|8430134.

In [8]:
# ────────────────────────────────────────────────────────────────────────────
# 5. PIVOT OPERATIONS
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("5. PIVOT OPERATIONS")
print("=" * 60)

# PIVOT - Payment methods by country
print("\n🔄 Pivot: Payment methods by country...")

payment_by_country = sales.groupBy("country") \
    .pivot("payment_method") \
    .count()

payment_by_country.show()

# PIVOT with aggregation - Average amount by payment method and country
print("\n🔄 Pivot: Average amount by payment method and country...")

avg_by_payment_country = sales.groupBy("country") \
    .pivot("payment_method") \
    .avg("final_amount") \
    .withColumnRenamed("Credit Card", "avg_credit") \
    .withColumnRenamed("Debit Card", "avg_debit") \
    .withColumnRenamed("PayPal", "avg_paypal")

avg_by_payment_country.show()


# ────────────────────────────────────────────────────────────────────────────
# 6. JSON & STRUCT OPERATIONS
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("6. WORKING WITH JSON & STRUCTS")
print("=" * 60)

# Create nested structure - User profile with stats
print("\n🏗️ Creating nested user profile...")

user_stats = sales.groupBy("user_id") \
    .agg(
        count("*").alias("total_orders"),
        sum("final_amount").alias("lifetime_value"),
        avg("final_amount").alias("avg_order_value"),
        collect_set("country").alias("countries")
    )

user_profile = users.join(user_stats, "user_id", "left") \
    .select(
        col("user_id"),
        struct(
            col("first_name"),
            col("last_name"),
            col("email")
        ).alias("personal_info"),
        struct(
            col("age"),
            col("country"),
            col("city")
        ).alias("location"),
        struct(
            col("total_orders"),
            col("lifetime_value"),
            col("avg_order_value"),
            col("countries")
        ).alias("purchase_stats")
    )

print("User profiles with nested structure:")
user_profile.show(5, truncate=False)

# Access nested fields
print("\n📋 Accessing nested fields - Flattened view:")

flattened_users = user_profile.select(
    col("user_id"),
    col("personal_info.first_name").alias("first_name"),
    col("personal_info.last_name").alias("last_name"),
    col("location.country").alias("country"),
    col("purchase_stats.lifetime_value").alias("ltv"),
    col("purchase_stats.total_orders").alias("orders")
)

flattened_users.show(10)

# Convert to JSON string
print("\n📝 Converting to JSON...")

user_json = user_profile \
    .withColumn("profile_json", to_json(struct("*")))

user_json.select("user_id", "profile_json").show(10, truncate=False)



5. PIVOT OPERATIONS

🔄 Pivot: Payment methods by country...


+---------+---------+-------------+-----------+----------+----------+------+
|  country|Apple Pay|Bank Transfer|Credit Card|Debit Card|Google Pay|PayPal|
+---------+---------+-------------+-----------+----------+----------+------+
|  Germany|   416252|       415913|     416873|    417192|    416720|417512|
|   France|   416655|       416559|     417603|    416539|    416389|416750|
|      USA|   416299|       417271|     416865|    416330|    415660|416016|
|       UK|   416923|       416261|     417406|    416876|    416362|417174|
|   Canada|   417462|       417207|     416911|    417014|    416641|416651|
|   Brazil|   416815|       416047|     416652|    416406|    416724|418450|
|    Japan|   417761|       415382|     415821|    415987|    416608|416708|
|Australia|   416167|       415185|     416633|    416250|    416681|417437|
+---------+---------+-------------+-----------+----------+----------+------+


🔄 Pivot: Average amount by payment method and country...


+---------+------------------+------------------+------------------+------------------+------------------+------------------+
|  country|         Apple Pay|     Bank Transfer|        avg_credit|         avg_debit|        Google Pay|        avg_paypal|
+---------+------------------+------------------+------------------+------------------+------------------+------------------+
|  Germany|458.40786374599907| 459.5347024497918|459.92221235244295| 458.9577795835005| 460.1198408283752|   459.30081501849|
|   France|458.52470499574014| 459.6687148279119|459.19069024887256| 459.0993787136375| 458.0770601288699|459.93855786442623|
|      USA|459.16221590731755|459.81527506105203| 459.0506932460145|459.29596387480973| 459.1077545590153|459.37267977673906|
|       UK|459.48564377594806|458.51746694982313| 459.0642674757911|  459.799394880011|459.19519242870433|459.24333139169823|
|   Canada|459.61844840967564| 459.1149542792913|459.18107500161875|459.28320665493334|  458.685235850529|459.28104567

+-------+--------------------------------------------+-----------------------+---------------------------------------------------------------------------------+
|user_id|personal_info                               |location               |purchase_stats                                                                   |
+-------+--------------------------------------------+-----------------------+---------------------------------------------------------------------------------+
|1000005|{Donald, Rodriguez, user1000005@example.com}|{33, USA, Houston}     |{9, 2969.32, 329.9244444444445, [France, Australia, Brazil, Germany, Japan, USA]}|
|2000003|{Nancy, Hernandez, user2000003@example.com} |{52, Germany, Munich}  |{2, 640.44, 320.22, [Australia, Canada]}                                         |
|2696460|{Jessica, Scott, user2696460@example.com}   |{48, Australia, Sydney}|{6, 3406.5299999999997, 567.755, [Australia, Brazil, Canada, USA]}               |
|3000003|{Thomas, Walker, user3000

+-------+-----------+---------+---------+------------------+------+
|user_id| first_name|last_name|  country|               ltv|orders|
+-------+-----------+---------+---------+------------------+------+
|1000005|     Donald|Rodriguez|      USA|           2969.32|     9|
|1696461|     Joseph|    Brown|Australia|           3639.65|    10|
|1696462|      James|    Perez|Australia|3602.0299999999997|     4|
|1696463|     Thomas|    Allen|   Canada|            621.71|     4|
|2000003|      Nancy|Hernandez|  Germany|            640.44|     2|
|2696460|    Jessica|    Scott|Australia|3406.5299999999997|     6|
|2696466|     Andrew|   Garcia|       UK|           1048.51|     3|
|3000003|     Thomas|   Walker|   Brazil|            276.99|     1|
|      1|     Robert|Rodriguez|    Japan|           4900.59|     7|
|      3|Christopher|    Jones|       UK|3056.2499999999995|    10|
+-------+-----------+---------+---------+------------------+------+
only showing top 10 rows


📝 Converting to JSON.

[Stage 139:>                                                        (0 + 2) / 2]

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|profile_json                                                                                                                                                                                                                                                                                                                                     |
+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
# ────────────────────────────────────────────────────────────────────────────
# PERFORMANCE COMPARISON: Regular UDF vs Pandas UDF
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("PERFORMANCE: Regular UDF vs Pandas UDF")
print("=" * 60)

# Create test data from your actual sales
test_df = sales.select("final_amount").limit(100000)

# Regular UDF
@udf(StringType())
def regular_tier(amount):
    if amount > 1000:
        return "Premium"
    elif amount > 500:
        return "High"
    elif amount > 100:
        return "Medium"
    else:
        return "Low"

# Pandas UDF
@pandas_udf(StringType())
def pandas_tier(amounts: pd.Series) -> pd.Series:
    conditions = [
        (amounts > 1000, "Premium"),
        (amounts > 500, "High"),
        (amounts > 100, "Medium"),
        (amounts > 0, "Low")
    ]
    
    result = pd.Series(["Low"] * len(amounts))
    for condition, label in conditions:
        result[condition] = label
    
    return result

print(f"\n📊 Testing with {test_df.count():,} rows...")

# Test regular UDF
start = time.time()
test_df.withColumn("tier", regular_tier(col("final_amount"))).count()
regular_time = time.time() - start
print(f"Regular UDF: {regular_time:.2f} seconds")

# Test Pandas UDF
start = time.time()
test_df.withColumn("tier", pandas_tier(col("final_amount"))).count()
pandas_time = time.time() - start
print(f"Pandas UDF: {pandas_time:.2f} seconds")

if regular_time > 0:
    print(f"\n💡 Pandas UDF is {regular_time/pandas_time:.1f}x faster!")

    


PERFORMANCE: Regular UDF vs Pandas UDF



📊 Testing with 100,000 rows...
Regular UDF: 0.73 seconds
Pandas UDF: 0.79 seconds

💡 Pandas UDF is 0.9x faster!


In [10]:
# ────────────────────────────────────────────────────────────────────────────
# COMPLEX BUSINESS EXAMPLE: Customer 360 View
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("🎯 COMPLEX EXAMPLE: Customer 360 View")
print("=" * 60)

# Combine all transformations into one comprehensive customer view
print("\n📊 Building Customer 360 view...")

# 1. Get customer base info
customer_base = users.select("user_id", "first_name", "last_name", "country", "age")

# 2. Get sales stats
sales_stats = sales.groupBy("user_id").agg(
    count("*").alias("total_transactions"),
    sum("final_amount").alias("total_spent"),
    avg("final_amount").alias("avg_transaction"),
    max("final_amount").alias("max_transaction"),
    min("final_amount").alias("min_transaction"),
    countDistinct("payment_method").alias("payment_methods_used"),
    collect_set("product_id").alias("products_purchased")
)

# 3. Join everything
customer_360 = customer_base.join(sales_stats, "user_id", "left") \
    .fillna(0, subset=["total_transactions", "total_spent"])

# 4. Add derived columns with UDFs and transformations
customer_360 = customer_360 \
    .withColumn("customer_segment",
        when(col("total_spent") > 1000, "VIP")
        .when(col("total_spent") > 500, "Gold")
        .when(col("total_spent") > 100, "Silver")
        .otherwise("Bronze")
    ) \
    .withColumn("avg_order_value", 
        when(col("total_transactions") > 0, 
             col("total_spent") / col("total_transactions"))
        .otherwise(0)
    ) \
    .withColumn("age_group",
        when(col("age") < 25, "18-24")
        .when(col("age") < 35, "25-34")
        .when(col("age") < 50, "35-49")
        .otherwise("50+")
    ) \
    .withColumn("product_diversity", size(col("products_purchased")))

print("Customer 360 view sample:")
customer_360.select(
    "user_id", "first_name", "last_name", "country",
    "total_transactions", "total_spent", "customer_segment",
    "age_group", "product_diversity"
).show(10)

# Summary by segment
print("\n📊 Customer Segment Summary:")
customer_360.groupBy("customer_segment").agg(
    count("*").alias("customer_count"),
    avg("total_spent").alias("avg_spent"),
    sum("total_spent").alias("total_revenue"),
    avg("product_diversity").alias("avg_products")
).orderBy("avg_spent").show()




🎯 COMPLEX EXAMPLE: Customer 360 View

📊 Building Customer 360 view...
Customer 360 view sample:


+-------+----------+---------+-------+------------------+------------------+----------------+---------+-----------------+
|user_id|first_name|last_name|country|total_transactions|       total_spent|customer_segment|age_group|product_diversity|
+-------+----------+---------+-------+------------------+------------------+----------------+---------+-----------------+
|     12|  Michelle|   Taylor|    USA|                 5|           1226.77|             VIP|    18-24|                5|
|     26|   Kenneth|    Moore| Brazil|                10|           4543.73|             VIP|    25-34|               10|
|     27|   Barbara|   Miller|     UK|                11|           5325.91|             VIP|      50+|               11|
|     28|    Donald| Gonzalez| Canada|                 7|            2267.8|             VIP|    18-24|                7|
|     31|   Charles|  Ramirez|     UK|                11|            5334.2|             VIP|    35-49|               11|
|     34| Elizabeth|  Sa

[Stage 158:================================================>        (6 + 1) / 7]

+----------------+--------------+------------------+--------------------+-------------------+
|customer_segment|customer_count|         avg_spent|       total_revenue|       avg_products|
+----------------+--------------+------------------+--------------------+-------------------+
|          Bronze|        418143| 9.998058798066696|   4180618.300000002|-0.6430575186000962|
|          Silver|        463352| 313.4474681883325|1.4523651128000024E8| 1.3951855177057615|
|            Gold|        499447| 751.2131535478259|  3.75191155900001E8|   2.31406535628405|
|             VIP|       2619058|3305.9006760942525| 8.658345612930061E9|  6.918965139374539|
+----------------+--------------+------------------+--------------------+-------------------+



In [12]:
# ────────────────────────────────────────────────────────────────────────────
# BEST PRACTICES SUMMARY
# ────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("UDF & TRANSFORMATION BEST PRACTICES")
print("=" * 60)

print("""
📌 KEY PRINCIPLES:
───────────────────────────────────────────────────────────────
1. BUILT-IN FUNCTIONS FIRST
   ✓ Always check if Spark has a built-in function for your need
   ✓ Built-in functions are optimized by Catalyst
   ✓ Examples: when(), otherwise(), regexp_extract(), etc.

2. UDFs AS LAST RESORT
   ✓ Use only when built-in functions can't express your logic
   ✓ Regular UDFs are slow (row-by-row, serialization overhead)
   ✓ Pandas UDFs are 10-100x faster than regular UDFs

3. TRANSFORMATION ORDER MATTERS
   ✓ Filter early, filter often
   ✓ Reduce data size before joins and aggregations
   ✓ Use broadcast for small tables

4. WINDOW FUNCTIONS
   ✓ Great for rankings, running totals, moving averages
   ✓ More efficient than self-joins
   ✓ Understand partitionBy and orderBy

5. COMPLEX DATA TYPES
   ✓ Arrays, Maps, Structs can reduce shuffle
   ✓ Use explode() when you need rows from arrays
   ✓ JSON functions for semi-structured data

6. PIVOT IS EXPENSIVE
   ✓ Know the distinct values before pivoting
   ✓ Can cause massive shuffle if many distinct values
   ✓ Consider alternative approaches for high cardinality

7. PERFORMANCE TIPS
   ✓ Use explain() to see execution plans
   ✓ Monitor Spark UI (port 4040)
   ✓ Cache intermediate results used multiple times
   ✓ Right-size your partitions
""")

spark.stop()

print("\n" + "=" * 60)
print("✅ ADVANCED TRANSFORMATIONS & UDFs COMPLETE")
print("=" * 60)






UDF & TRANSFORMATION BEST PRACTICES

📌 KEY PRINCIPLES:
───────────────────────────────────────────────────────────────
1. BUILT-IN FUNCTIONS FIRST
   ✓ Always check if Spark has a built-in function for your need
   ✓ Built-in functions are optimized by Catalyst
   ✓ Examples: when(), otherwise(), regexp_extract(), etc.

2. UDFs AS LAST RESORT
   ✓ Use only when built-in functions can't express your logic
   ✓ Regular UDFs are slow (row-by-row, serialization overhead)
   ✓ Pandas UDFs are 10-100x faster than regular UDFs

3. TRANSFORMATION ORDER MATTERS
   ✓ Filter early, filter often
   ✓ Reduce data size before joins and aggregations
   ✓ Use broadcast for small tables

4. WINDOW FUNCTIONS
   ✓ Great for rankings, running totals, moving averages
   ✓ More efficient than self-joins
   ✓ Understand partitionBy and orderBy

5. COMPLEX DATA TYPES
   ✓ Arrays, Maps, Structs can reduce shuffle
   ✓ Use explode() when you need rows from arrays
   ✓ JSON functions for semi-structured data

6